In [ ]:
import sys, random, importlib
sys.path.insert(0, "..")
import torch

# Reload to pick up any in-session changes to sorl_trainer
import sorl.sorl_trainer as _st; importlib.reload(_st)
from sorl.sorl_trainer import sorl_search, infer_insert_mask, insert_tokens_with_padding
from sorl.trainer_ablate import _drop_nl_prefix_m_set
from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from data.pt_dataset import get_dataset, collate_fn

# ── Config ──────────────────────────────────────────────────────────────
MODEL_NAME  = "Qwen/Qwen3-0.6B"
ABS_VOCAB   = 32
K           = 4
N_SAMPLES   = 4
ANSWER_TOK  = 820   # "####" delimiter in GSM8K

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = SorlModelWrapper.from_pretrained(MODEL_NAME, abstract_vocab_size_list=[ABS_VOCAB])
model     = model.to(device).eval()

base_vocab = int(model.vocab_sizes[0].item())
pad_id     = tokenizer.pad_token_id

# ── GSM8K batch ──────────────────────────────────────────────────────────
ds         = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
batch      = collate_fn([ds[i] for i in range(N_SAMPLES)])
input_ids  = batch["input_ids"].to(device)
attn_mask  = batch["attention_mask"].to(device)
prompt_len = batch["prompt_len"].to(device)

# ── Decode helper ─────────────────────────────────────────────────────────
def decode_annotated(ids_1d, valid_len=None):
    """NL tokens → text, abstract tokens → [ABS]. Pass a 1-D id tensor."""
    n = valid_len if valid_len is not None else len(ids_1d)
    parts, buf = [], []
    for tid in ids_1d[:n].tolist():
        if tid >= base_vocab:
            if buf: parts.append(tokenizer.decode(buf, skip_special_tokens=False)); buf = []
            parts.append("[ABS]")
        else:
            buf.append(tid)
    if buf: parts.append(tokenizer.decode(buf, skip_special_tokens=False))
    return "".join(parts)

print(f"device={device}  base_vocab={base_vocab}  abs_vocab={ABS_VOCAB}  K={K}")
print(f"GSM8K lens: {[int(attn_mask[b].sum()) for b in range(N_SAMPLES)]}")

In [ ]:
# ── Premise: [ABS] tokens appear ONLY in CoT, never in query or answer ──────
# cot_only_abs=True in sorl_search:
#   • infer_insert_mask gates out prompt positions  (no ABS in query)
#   • also gates out positions at/after ANSWER_TOK  (no ABS in answer)
#
# Resulting layout:
#   [Q_text ... Q_text]  [CoT [ABS] CoT [ABS] ...]  [#### ans_text]
#                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^ ABS only here

with torch.no_grad():
    best_data, _, _, exp_attn, exp_pl = sorl_search(
        model, input_ids, attn_mask, prompt_len, pad_id,
        n=2, K=K, max_iterations=2,
        memory_span_abs=1792, memory_span_traj=1792,
        temperature=1.0,
        cot_only_abs=True,          # ← key flag
    )

print("=== sorl_search  (cot_only_abs=True) ===\n")
for b in range(N_SAMPLES):
    valid = int(exp_attn[b].sum())
    pl    = exp_pl[b].item()
    seq   = best_data[b, :valid]

    query = seq[:pl]
    resp  = seq[pl:]

    # split response at ####
    ans_positions = (resp == ANSWER_TOK).nonzero(as_tuple=True)[0]
    if len(ans_positions):
        ai   = ans_positions[0].item()
        cot  = resp[:ai]
        ans  = resp[ai:]
    else:
        cot, ans = resp, resp[:0]

    n_q   = (query >= base_vocab).sum().item()
    n_cot = (cot   >= base_vocab).sum().item()
    n_ans = (ans   >= base_vocab).sum().item()

    q_text   = tokenizer.decode([t for t in query.tolist() if t < base_vocab], skip_special_tokens=True)
    cot_text = decode_annotated(cot)
    ans_text = tokenizer.decode([t for t in ans.tolist()   if t < base_vocab], skip_special_tokens=False)

    print(f"[{b}]  total={valid}  ABS→  query:{n_q}  CoT:{n_cot}  answer:{n_ans}")
    print(f"  Q  : {q_text[:100]}")
    print(f"  CoT: {cot_text[:200]}")
    print(f"  Ans: {ans_text.strip()[:60]}")
    print()

=== ORIGINAL SEQUENCES ===
  batch[0]  len=26           nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] nl108 nl109 nl110 nl111 [A3] nl112 nl113 nl114 nl115 [A4] [####] nl200 nl201
  batch[1]  len=16           nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] [####] nl200 nl201


In [ ]:
# ── NL Replacement trick: _drop_nl_prefix_m_set ─────────────────────────────
# Starting from the cot_only_abs expanded sequences (best_data above):
#   • Sample m from M_SET per sequence independently
#   • Drop the first m NL tokens from the CoT prefix
#   • Keep the [ABS] tokens that summarised those dropped NL spans
#   • The suffix CoT (from NL m+1 onward) and the answer are untouched
#
# Result layout (m>0):
#   [Q_text]  [[ABS]...[ABS]]  [CoT_suffix [ABS] ...]  [#### ans_text]
#               ^^^^^^^^^^^    ← abs from dropped prefix kept as prefix
#
# M_SET = (0, 16, 32, 64, 128): 0 = no compression, 128 = heavy compression

random.seed(42)
M_SET = (0, 16, 32, 64, 128)

print(f"M_SET = {M_SET}  (m sampled per-sequence each trial)\n")

for trial in range(4):
    print(f"{'─'*72}  trial {trial + 1}")
    out_ids, out_attn, out_pl = _drop_nl_prefix_m_set(
        best_data, exp_attn, exp_pl,
        base_vocab, pad_id, m_set=M_SET, answer_token_id=ANSWER_TOK,
    )
    for b in range(N_SAMPLES):
        valid = int(out_attn[b].sum())
        pl    = out_pl[b].item()
        seq   = out_ids[b, :valid]
        resp  = seq[pl:]

        ans_positions = (resp == ANSWER_TOK).nonzero(as_tuple=True)[0]
        if len(ans_positions):
            ai   = ans_positions[0].item()
            cot  = resp[:ai]
            ans  = resp[ai:]
        else:
            cot, ans = resp, resp[:0]

        n_abs_cot = (cot >= base_vocab).sum().item()
        n_abs_ans = (ans >= base_vocab).sum().item()
        orig_len  = int(exp_attn[b].sum())

        cot_text = decode_annotated(cot)
        ans_text = tokenizer.decode([t for t in ans.tolist() if t < base_vocab],
                                    skip_special_tokens=False)

        print(f"  [{b}]  {orig_len}→{valid}tok  ABS_cot={n_abs_cot}  ABS_ans={n_abs_ans}")
        print(f"       CoT: {cot_text[:180]}")
        print(f"       Ans: {ans_text.strip()[:50]}")
    print()


──────────────────────────────────────────────────────────────────────
  m=0 only (no compression)  →  sampled per batch item independently
──────────────────────────────────────────────────────────────────────
  trial 1:
    batch[0]  len=26         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] nl108 nl109 nl110 nl111 [A3] nl112 nl113 nl114 nl115 [A4] [####] nl200 nl201
    batch[1]  len=16         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] [####] nl200 nl201
  trial 2:
    batch[0]  len=26         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] nl108 nl109 nl110 nl111 [A3] nl112 nl113 nl114 nl115 [A4] [####] nl200 nl201
    batch[1]  len=16         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] [####] nl200 nl201
  trial 3:
    batch[0]  len=26         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] nl108 nl109 nl110 nl111 [A3] nl112 nl113 nl114 nl115 

(torch.Size([2, 26]), torch.Size([2, 26]))

device: cpu


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


base_vocab=151936  abs_vocab=32  pad_id=151643


=== RAW GSM8K SEQUENCES ===

[0] len=100  prompt_len=42
  Q  : Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips 
  CoT:  Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

[1] len=99  prompt_len=35
  Q  : Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she ea
  CoT:  Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

[2] len=164  prompt_len=64
  Q  : Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her paren
  CoT:  In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.
#### 5

[3] len=183  prompt_len=57
  Q  : Question

=== AFTER sorl_search (interleaved abstract tokens) ===

[0]  expanded_len=124  prompt_len=52
  Question: Natalia[ABS] sold clips to [ABS]48 of her[ABS] friends in April,[ABS] and then she sold[ABS] half as many clips[ABS] in May. How[ABS] many clips did Natal[ABS]ia sell altogether in[ABS] April and May?
[ABS]Answer: Natalia[ABS] sold 48[ABS]/2 = <<[ABS]48/2[ABS]=24>>[ABS]24 clips in[ABS] May.
Natal[ABS]ia sold 4[ABS]8+24[ABS] = <<48[ABS]+24

[1]  expanded_len=123  prompt_len=43
  Question: Weng[ABS] earns $12[ABS] an hour for babys[ABS]itting. Yesterday,[ABS] she just did [ABS]50 minutes of[ABS] babysitting. How[ABS] much did she earn[ABS]?
Answer: W[ABS]eng earns 1[ABS]2/60[ABS] = $<<1[ABS]2/60[ABS]=0.2[ABS]>>0.2[ABS] per minute.
Working[ABS] 50 minutes[ABS], she earned [ABS]0.2 x[ABS] 50 =[ABS] $<<0.[ABS]2*50[ABS]=10>>[AB

[2]  expanded_len=204  prompt_len=79
  Question: Betty is[ABS] saving money for a[ABS] new wallet which costs[ABS] $100[ABS]. Betty has only[ABS] half of the mon

=== AFTER _drop_nl_prefix_m_set  (M_SET=(0, 16, 32, 64, 128)) ===
CoT prefix NL tokens are dropped; their abstract tokens are kept.
'####' and the answer are always preserved.

──────────────────────────────────────────────────────────────────────  trial 1
  [0]  len 124 → 124  (−0 NL tokens dropped)
       Question: Natalia[ABS] sold clips to [ABS]48 of her[ABS] friends in April,[ABS] and then she sold[ABS] half as many clips[ABS] in May. How[ABS] many clips did Natal[ABS]ia sell altogether in[ABS] April and May?
[ABS]Answer: Natalia[ABS] sold 48[ABS]/2 = <<[ABS]48/2[ABS]=24>>[ABS]24 clips in[ABS] May
  [1]  len 123 → 123  (−0 NL tokens dropped)
       Question: Weng[ABS] earns $12[ABS] an hour for babys[ABS]itting. Yesterday,[ABS] she just did [ABS]50 minutes of[ABS] babysitting. How[ABS] much did she earn[ABS]?
Answer: W[ABS]eng earns 1[ABS]2/60[ABS] = $<<1[ABS]2/60[ABS]=0.2[ABS]>>0.2[ABS] per minute.
Working[ABS] 50 minutes[ABS], she earned [AB
  [2]  len 204 → 172  (−32 NL tokens 

TypeError: infer_insert_mask() got an unexpected keyword argument 'answer_token_id'